# Mature ChP Subclustering, Stress, and Mitochondrial Signature Analysis

This notebook starts after the broad GSE150903 cell-identity analysis. The goal is to isolate mature choroid plexus-like epithelial cells, rerun dimensionality reduction on that subset, and ask whether mature ChP contains finer sub-states such as ciliated/light, mitochondria-rich/dark, and myoepithelial-like populations.

This notebook also scores stress and mitochondrial gene programs. Because the input is a processed SCT-scaled matrix, this analysis uses gene-signature scores rather than raw mitochondrial read percentages.

## Scope Of This Notebook

This notebook focuses on:

1. loading the annotated full dataset from the previous notebook
2. selecting mature ChP-like cells
3. rerunning PCA on only the mature ChP subset
4. using PCs 1-12 for mature ChP neighbors/UMAP/Leiden clustering, following the paper's reported subclustering parameter
5. validating subclusters with mature ChP, ciliated/light, dark/mitochondria-rich, myoepithelial-like, barrier, and transport markers
6. scoring stress and mitochondrial gene signatures
7. saving the mature ChP subset and selected figures

This notebook does not perform GO/pathway enrichment or external human/mouse reference comparison. Those analyses should go in the next notebook.

## Key Assumptions

This notebook expects the previous methods-guided notebook to have saved:

```text
data/processed/methods_guided_reanalysis.h5ad
```

That object should contain:

- the processed expression matrix
- sample metadata in `adata.obs["sample"]`
- broad cluster labels in `adata.obs["leiden"]`
- a cell-type annotation column such as `cell_type_reviewed` or `cell_type_auto`

If your mature ChP labels have a different column name or wording, update the selection cell below before proceeding.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=120, frameon=False)

PROJECT_DIR = Path("/Users/Princess/Documents/Manju's Research Portfolio")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURE_DIR = PROJECT_DIR / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FULL_ADATA_PATH = PROCESSED_DIR / "methods_guided_reanalysis.h5ad"
MATURE_ADATA_PATH = PROCESSED_DIR / "mature_chp_subclustered_stress_mito.h5ad"

print("Project directory:", PROJECT_DIR)
print("Full annotated AnnData path:", FULL_ADATA_PATH)
print("Full annotated AnnData exists:", FULL_ADATA_PATH.exists())

## 1. Load Annotated Full Dataset

Load the full annotated AnnData object produced by the methods-guided notebook. If this file does not exist, first run notebook `02_methods_guided_reanalysis.ipynb` through the save step.

In [ ]:
if not FULL_ADATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {FULL_ADATA_PATH}. Run notebook 02 through the save step first."
    )

adata = sc.read_h5ad(FULL_ADATA_PATH)
print(adata)
print("obs columns:")
print(list(adata.obs.columns))

display(adata.obs.head())

## 2. Choose The Mature ChP Annotation Column

The mature ChP subset has to be selected from a broad cell-type annotation. This notebook looks for `cell_type_reviewed` first, then `cell_type_auto`.

Review the value counts before continuing. If the labels do not contain mature ChP-like wording, edit `label_column` and/or `mature_label_pattern` in the next cell.

In [ ]:
preferred_label_columns = ["cell_type_reviewed", "cell_type_auto", "cell_type", "annotation"]
label_column = next((col for col in preferred_label_columns if col in adata.obs.columns), None)

if label_column is None:
    raise KeyError(
        "No cell-type annotation column found. Expected one of: "
        + ", ".join(preferred_label_columns)
    )

print("Using label column:", label_column)
display(adata.obs[label_column].astype(str).value_counts())

## 3. Subset Mature ChP-Like Cells

The paper subclustered the mature ChP epithelial population after broad clustering. Here, we create a smaller AnnData object containing only cells whose broad annotation suggests mature ChP identity.

The default pattern is intentionally broad: it captures labels containing `Mature ChP` or `Choroid plexus`. If this selects too many or too few cells, update `mature_label_pattern`.

In [ ]:
mature_label_pattern = "Mature ChP|Choroid plexus|ChP epithelium"

mature_mask = (
    adata.obs[label_column]
    .astype(str)
    .str.contains(mature_label_pattern, case=False, regex=True, na=False)
)

adata_mature = adata[mature_mask].copy()

print("Mature/ChP-like cells selected:", adata_mature.n_obs)
print("Genes retained:", adata_mature.n_vars)

if adata_mature.n_obs == 0:
    print("Available labels:")
    print(adata.obs[label_column].astype(str).value_counts())
    raise ValueError("No mature ChP-like cells selected. Edit mature_label_pattern before continuing.")

display(adata_mature.obs[label_column].astype(str).value_counts())

if "sample" in adata_mature.obs.columns:
    display(adata_mature.obs["sample"].astype(str).value_counts())

## 4. Rerun PCA, Neighbors, UMAP, And Leiden On Mature ChP Cells

The paper reports that mature ChP subclustering used PCs 1-12, selected by ElbowPlot. These PCs are recalculated within the mature ChP subset; they are not reused from the full-dataset PCA.

In [ ]:
sc.tl.pca(adata_mature, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata_mature, log=True, n_pcs=30)

sc.pp.neighbors(
    adata_mature,
    n_neighbors=10,
    n_pcs=12,
)

sc.tl.umap(
    adata_mature,
    min_dist=0.2,
    random_state=42,
)

sc.tl.leiden(
    adata_mature,
    resolution=0.5,
    flavor="igraph",
    n_iterations=2,
    directed=False,
)

plot_colors = ["sample", "leiden"] if "sample" in adata_mature.obs.columns else ["leiden"]
sc.pl.umap(
    adata_mature,
    color=plot_colors,
    size=8,
    alpha=0.85,
    wspace=0.4,
    frameon=False,
    title=["Mature ChP subset by sample", "Mature ChP subclusters"][:len(plot_colors)],
)

print("Mature ChP subclustering complete.")

## 5. Four-Panel Sample Highlight UMAP For Mature ChP Subset

This view asks whether mature ChP-like cells from different samples occupy similar or distinct regions of the mature ChP UMAP. Overlap is biologically meaningful and may indicate shared mature ChP states across timepoints.

In [ ]:
if "sample" not in adata_mature.obs.columns:
    print("No sample column found; skipping sample-highlight plot.")
else:
    umap = adata_mature.obsm["X_umap"]
    sample_series = adata_mature.obs["sample"].astype(str)
    samples = list(adata_mature.obs["sample"].cat.categories) if hasattr(adata_mature.obs["sample"], "cat") else sorted(sample_series.unique())

    sample_colors = {
        "Choroid Plexus Org D27": "#1f77b4",
        "Choroid Plexus Org D46": "#ff7f0e",
        "Choroid Plexus Org D53": "#2ca02c",
        "Telencephalon organoids D55": "#d62728",
    }

    ncols = 2
    nrows = int(np.ceil(len(samples) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(10, 4.5 * nrows), sharex=True, sharey=True)
    axes = np.array(axes).reshape(-1)

    x_pad = (umap[:, 0].max() - umap[:, 0].min()) * 0.05
    y_pad = (umap[:, 1].max() - umap[:, 1].min()) * 0.05
    xlim = (umap[:, 0].min() - x_pad, umap[:, 0].max() + x_pad)
    ylim = (umap[:, 1].min() - y_pad, umap[:, 1].max() + y_pad)

    for ax, sample in zip(axes, samples):
        mask = sample_series.to_numpy() == str(sample)
        color = sample_colors.get(str(sample), "#1f77b4")
        ax.scatter(umap[~mask, 0], umap[~mask, 1], s=2, c="#d0d0d0", alpha=0.2, linewidths=0, rasterized=True)
        ax.scatter(umap[mask, 0], umap[mask, 1], s=4, c=color, alpha=0.9, linewidths=0, rasterized=True)
        ax.set_title(str(sample), fontsize=12)
        ax.set_xlabel("UMAP1")
        ax.set_ylabel("UMAP2")
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.grid(False)

    for ax in axes[len(samples):]:
        ax.set_visible(False)

    fig.suptitle("Mature ChP Sample Distribution", fontsize=16, y=0.98)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "mature_chp_sample_highlight_umap.png", bbox_inches="tight", dpi=300)
    plt.show()

## 6. Validate Mature ChP Subclusters With Marker Sets

This section checks whether mature ChP subclusters show marker patterns consistent with the paper's described ChP epithelial states.

Interpretation guide:

- `TTR`, `AQP1`, `KRT18`, `NME5`: mature ChP epithelial identity
- `FOXJ1`, `ARL13B`, `CCDC67`: light/ciliated ChP-like state
- `CARD19`, `IGF2`, `RBP1`: dark/mitochondria-rich ChP-like state
- `KRT17`, `ACTA2`, `TAGLN`: myoepithelial-like state
- `CLDN1`, `CLDN3`, `TJP1`, `AQP1`, `CA2`, `SLC23A2`: barrier/transport-associated genes

In [ ]:
mature_chp_marker_sets = {
    "Mature ChP epithelium": ["TTR", "AQP1", "KRT18", "NME5"],
    "Light / ciliated ChP": ["FOXJ1", "ARL13B", "CCDC67"],
    "Dark / mitochondria-rich ChP": ["CARD19", "IGF2", "RBP1"],
    "Myoepithelial-like ChP": ["KRT17", "ACTA2", "TAGLN"],
    "Barrier / tight junction": ["CLDN1", "CLDN3", "CLDN5", "TJP1", "TJP2", "OCLN"],
    "CSF secretion / transport": ["AQP1", "CA2", "CA12", "SLC23A2", "SLC46A1"],
    "Cycling": ["MKI67", "TOP2A", "PCNA"],
}

mature_chp_marker_sets_present = {
    group: [gene for gene in genes if gene in adata_mature.var_names]
    for group, genes in mature_chp_marker_sets.items()
}
mature_chp_marker_sets_present = {
    group: genes for group, genes in mature_chp_marker_sets_present.items() if genes
}

print("Marker sets present in mature ChP subset:")
for group, genes in mature_chp_marker_sets_present.items():
    print(f"{group}: {genes}")

sc.pl.dotplot(
    adata_mature,
    var_names=mature_chp_marker_sets_present,
    groupby="leiden",
    standard_scale="var",
    dendrogram=False,
)

plt.savefig(FIGURE_DIR / "mature_chp_marker_dotplot_by_leiden.png", bbox_inches="tight", dpi=300)

## 7. Rank Marker Genes Within Mature ChP Subclusters

This ranks genes that distinguish mature ChP subclusters from one another. Because the matrix is SCT-scaled, use these rankings as exploratory marker evidence and validate important genes with marker plots and known biology.

In [ ]:
sc.tl.rank_genes_groups(
    adata_mature,
    groupby="leiden",
    method="wilcoxon",
)

sc.pl.rank_genes_groups(
    adata_mature,
    n_genes=10,
    sharey=False,
)

mature_marker_table = sc.get.rank_genes_groups_df(adata_mature, group=None)
mature_marker_table.to_csv(PROCESSED_DIR / "mature_chp_subcluster_marker_genes.csv", index=False)
display(mature_marker_table.head(30))

## 8. Score Stress, Mitochondrial, And ChP State Gene Programs

This section calculates per-cell scores for biologically meaningful gene sets. Since this is not raw count data, the mitochondrial score here is a mitochondrial gene-expression signature, not a raw mitochondrial read percentage.

In [ ]:
signature_sets = {
    "stress_score": [
        "FOS", "JUN", "JUNB", "JUND", "ATF3", "DDIT3",
        "HSPA1A", "HSPA1B", "HSP90AA1", "DNAJB1",
    ],
    "mitochondrial_gene_score": [
        "MT-CO1", "MT-CO2", "MT-CO3", "MT-ND1", "MT-ND2", "MT-ND3",
        "MT-ND4", "MT-ND5", "MT-CYB", "MT-ATP6", "MT-ATP8",
    ],
    "dark_mito_chp_score": ["CARD19", "IGF2", "RBP1"],
    "light_ciliated_chp_score": ["FOXJ1", "ARL13B", "CCDC67"],
    "myoepithelial_like_score": ["KRT17", "ACTA2", "TAGLN"],
    "barrier_transport_score": ["CLDN1", "CLDN3", "TJP1", "AQP1", "CA2", "SLC23A2"],
}

score_columns = []
for score_name, genes in signature_sets.items():
    genes_present = [gene for gene in genes if gene in adata_mature.var_names]
    print(f"{score_name}: {len(genes_present)} / {len(genes)} genes present -> {genes_present}")
    if len(genes_present) == 0:
        print(f"Skipping {score_name}; no genes present in matrix.")
        continue
    sc.tl.score_genes(
        adata_mature,
        gene_list=genes_present,
        score_name=score_name,
    )
    score_columns.append(score_name)

print("Scores added:", score_columns)

## 9. Visualize Signature Scores On UMAP

These plots help ask whether stress, mitochondrial, or dark/ciliated ChP programs localize to specific mature ChP subclusters.

In [ ]:
if score_columns:
    sc.pl.umap(
        adata_mature,
        color=score_columns,
        cmap="viridis",
        size=8,
        frameon=False,
        ncols=3,
    )
else:
    print("No score columns available to plot.")

## 10. Compare Signature Scores Across Subclusters And Samples

These summaries help determine whether a signature is subcluster-specific, sample-specific, or broadly distributed.

In [ ]:
if score_columns:
    cluster_score_summary = adata_mature.obs.groupby("leiden", observed=True)[score_columns].mean()
    display(cluster_score_summary)
    cluster_score_summary.to_csv(PROCESSED_DIR / "mature_chp_signature_scores_by_leiden.csv")

    ax = cluster_score_summary.plot(kind="bar", figsize=(12, 5))
    ax.set_ylabel("Mean signature score")
    ax.set_xlabel("Mature ChP Leiden subcluster")
    ax.set_title("Mean Signature Scores By Mature ChP Subcluster")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "mature_chp_signature_scores_by_leiden.png", bbox_inches="tight", dpi=300)
    plt.show()

    if "sample" in adata_mature.obs.columns:
        sample_score_summary = adata_mature.obs.groupby("sample", observed=True)[score_columns].mean()
        display(sample_score_summary)
        sample_score_summary.to_csv(PROCESSED_DIR / "mature_chp_signature_scores_by_sample.csv")
else:
    print("No score columns available to summarize.")

## 11. Interpretation Checkpoint

Before moving to enrichment or reference comparison, review:

- Which mature ChP subclusters express `CARD19`, `IGF2`, and `RBP1`?
- Are ciliated/light markers such as `FOXJ1`, `ARL13B`, and `CCDC67` localized to a specific subcluster?
- Are stress signatures concentrated in one subcluster or one sample?
- Does the mitochondrial gene score match the dark/mitochondria-rich ChP marker score, or do they separate?
- Are any subclusters mostly from a single timepoint, suggesting developmental differences?

If the subcluster labels are unclear, adjust Leiden resolution and rerun marker validation before continuing.

## 12. Save Mature ChP Subset

Save the mature ChP subset with subcluster labels and signature scores for the next notebook.

In [ ]:
adata_mature.write_h5ad(MATURE_ADATA_PATH)
print("Saved mature ChP subset:", MATURE_ADATA_PATH)
print(adata_mature)